In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece protobuf huggingface_hub pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.4 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import login
import getpass

token = getpass.getpass("Paste your HuggingFace token (Enter to skip if only testing Qwen2.5): ")
if token.strip():
    login(token=token.strip())

Paste your HuggingFace token (Enter to skip if only testing Qwen2.5): ··········


In [ ]:
import gc
import re
import time
import json
import torch
import pandas as pd

from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODELS = {
    "Llama-1B": "meta-llama/Llama-3.2-1B-Instruct",
    "Llama-3B": "meta-llama/Llama-3.2-3B-Instruct",
    "Llama-8B": "meta-llama/Llama-3.1-8B-Instruct",
    "Qwen2.5-1.5B": "Qwen/Qwen2.5-1.5B-Instruct",
    "Qwen2.5-3B": "Qwen/Qwen2.5-3B-Instruct",
    "Qwen2.5-7B": "Qwen/Qwen2.5-7B-Instruct",
}

BNB_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

SYSTEM_PROMPT = """You are a factual research assistant. You will be given a user question and numbered source excerpts.

Rules:
- Answer ONLY using the provided sources. Do not use outside knowledge.
- Cite every factual claim with [n] matching the source number.
- If sources conflict, say so explicitly and present both sides.
- If sources don't cover the question, say "I don't have enough current information to answer this confidently" rather than guessing.
- Be concise. No filler, no restating the question."""

In [ ]:
def build_context(sources):
    if not sources:
        return "(no sources found)"
    blocks = []
    for i, s in enumerate(sources, 1):
        date_str = f" ({s['published_date']})" if s.get("published_date") else ""
        blocks.append(f"[{i}] {s['title']}{date_str}\n{s['text']}")
    return "\n\n".join(blocks)


def build_user_prompt(question, sources):
    return f"Sources:\n{build_context(sources)}\n\nQuestion: {question}"

In [ ]:
TEST_CASES = [
    {
        "name": "citation_single_source",
        "question": "Who created Python?",
        "sources": [
            {"title": "Python History", "published_date": None,
             "text": "Python was created by Guido van Rossum and first released in 1991."},
        ],
        "check_type": "grounded_fact",
        "expected_substrings": ["guido van rossum"],
    },
    {
        "name": "citation_pick_right_source",
        "question": "What year was Python first released?",
        "sources": [
            {"title": "Python Design Origins", "published_date": None,
             "text": "Python's design began in the late 1980s as a successor to the ABC language."},
            {"title": "Python Release History", "published_date": None,
             "text": "Python was first released to the public in 1991."},
        ],
        "check_type": "grounded_fact",
        "expected_substrings": ["1991"],
    },
    {
        "name": "conflicting_sources",
        "question": "Did the company's quarterly revenue increase?",
        "sources": [
            {"title": "Q2 Earnings Release", "published_date": None,
             "text": "Company X reported a 12% increase in quarterly revenue, beating analyst expectations."},
            {"title": "Internal Filing Summary", "published_date": None,
             "text": "According to internal filings, Company X's quarterly revenue declined by 5%."},
        ],
        "check_type": "conflict",
    },
    {
        "name": "insufficient_information",
        "question": "What is the population of the newly founded city of Arcadia Falls?",
        "sources": [
            {"title": "Regional Tourism Guide", "published_date": None,
             "text": "The region surrounding Arcadia Falls is known for its mountain trails and ski resorts."},
        ],
        "check_type": "refusal",
    },
    {
        "name": "hallucination_resistance",
        "question": "Who is the CEO of Zylora Robotics?",
        "sources": [
            {"title": "Robotics Industry Overview", "published_date": None,
             "text": "The robotics industry has seen rapid growth, with several startups entering the humanoid robotics space in 2026."},
        ],
        "check_type": "refusal",
    },
    {
        "name": "noise_robustness",
        "question": "What is the capital of France?",
        "sources": [
            {"title": "Dog Facts", "published_date": None,
             "text": "Dogs have an excellent sense of smell, up to 40 times stronger than humans."},
            {"title": "European Capitals", "published_date": None,
             "text": "Paris is the capital and most populous city of France."},
            {"title": "Cat Facts", "published_date": None,
             "text": "Cats sleep for an average of 12 to 16 hours per day."},
        ],
        "check_type": "grounded_fact",
        "expected_substrings": ["paris"],
    },
    {
        "name": "multi_source_synthesis",
        "question": "Which country is the city of Kyoto in, and what is Japan known for?",
        "sources": [
            {"title": "City Profile: Kyoto", "published_date": None,
             "text": "Kyoto was the imperial capital of Japan for over a thousand years."},
            {"title": "Japan Overview", "published_date": None,
             "text": "Japan is an island nation in East Asia known for blending traditional and modern culture."},
        ],
        "check_type": "multi_cite",
        "expected_substrings": ["japan"],
    },
    {
        "name": "messy_real_world_context",
        "question": "Who is the current UN Secretary-General?",
        "sources": [
            {"title": "About the Secretary-General", "published_date": None,
             "text": ("# United Nations Secretary-General. Portrait of the United Nations "
                       "Secretary-General Ant\u00f3nio Guterres. Ant\u00f3nio Guterres, the ninth "
                       "Secretary-General of the United Nations, took office on 1 January 2017 "
                       "[Menu] [About] [Contact] [Press]")},
        ],
        "check_type": "grounded_fact",
        "expected_substrings": ["guterres"],
    },
    {
        "name": "zero_sources",
        "question": "What is today's weather in a city that doesn't exist?",
        "sources": [],
        "check_type": "refusal",
    },
]

In [ ]:
CITATION_RE = re.compile(r"\[(\d+)\]")

REFUSAL_KEYWORDS = [
    "don't have enough",
    "do not have enough",
    "not enough information",
    "insufficient information",
    "cannot determine",
    "cannot answer",
    "can't answer confidently",
]

CONFLICT_KEYWORDS = [
    "conflict", "disagree", "differ", "contradict",
    "however", "on the other hand", "mixed", "inconsistent", "discrepancy",
]


def score_response(case, response):
    r = response.lower()
    check_type = case["check_type"]
    citations_found = set(CITATION_RE.findall(response))

    if check_type == "grounded_fact":
        fact_ok = all(sub in r for sub in case["expected_substrings"])
        cited = len(citations_found) >= 1
        return fact_ok and cited

    elif check_type == "multi_cite":
        fact_ok = all(sub in r for sub in case["expected_substrings"])
        cited_multi = len(citations_found) >= 2
        return fact_ok and cited_multi

    elif check_type == "conflict":
        return any(k in r for k in CONFLICT_KEYWORDS)

    elif check_type == "refusal":
        return any(k in r for k in REFUSAL_KEYWORDS)

    return False

In [ ]:
def load_model(model_path):
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        quantization_config=BNB_CONFIG,
        device_map="auto",
    )
    model.eval()
    return tokenizer, model


def generate(tokenizer, model, question, sources, max_new_tokens=220):
    user_prompt = build_user_prompt(question, sources)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    start = time.perf_counter()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )
    latency = time.perf_counter() - start

    input_len = inputs["input_ids"].shape[1]
    new_tokens = outputs[0][input_len:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    return answer, latency, len(new_tokens)


def cleanup():
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.synchronize()

In [ ]:
def benchmark_model(model_name, model_path):
    print(f"\n{'='*70}\nLoading {model_name} ({model_path})\n{'='*70}")

    tokenizer, model = load_model(model_path)

    rows = []
    total_score = 0
    total_latency = 0
    total_tokens = 0

    for case in tqdm(TEST_CASES, desc=model_name):
        answer, latency, tokens = generate(tokenizer, model, case["question"], case["sources"])
        passed = score_response(case, answer)

        total_score += int(passed)
        total_latency += latency
        total_tokens += tokens

        print(f"\n--- {case['name']} [{'PASS' if passed else 'FAIL'}] ---")
        print(f"Q: {case['question']}")
        print(f"A: {answer}")
        print(f"[{tokens} tokens, {latency:.1f}s]")

        rows.append({
            "model": model_name,
            "case": case["name"],
            "check_type": case["check_type"],
            "question": case["question"],
            "response": answer,
            "pass": passed,
            "latency_sec": round(latency, 2),
            "tokens": tokens,
        })

    summary = {
        "model": model_name,
        "score_pct": round(total_score / len(TEST_CASES) * 100, 1),
        "avg_latency_sec": round(total_latency / len(TEST_CASES), 2),
        "avg_tokens": round(total_tokens / len(TEST_CASES), 1),
    }

    del model
    cleanup()
    return rows, summary

In [ ]:
import traceback

all_rows = []
summary_rows = []

for name, path in MODELS.items():
    try:
        rows, summary = benchmark_model(name, path)
        all_rows.extend(rows)
        summary_rows.append(summary)
    except Exception as e:
        print(f"\nFAILED to run {name}: {repr(e)}")
        traceback.print_exc()
        summary_rows.append({"model": name, "score_pct": None, "avg_latency_sec": None, "avg_tokens": None})
    finally:
        cleanup()

details_df = pd.DataFrame(all_rows)
summary_df = pd.DataFrame(summary_rows).sort_values("score_pct", ascending=False)

details_df.to_csv("detailed_results_v2.csv", index=False)
summary_df.to_csv("summary_results_v2.csv", index=False)

print("\nSaved detailed_results_v2.csv and summary_results_v2.csv")
summary_df


Loading Llama-1B (meta-llama/Llama-3.2-1B-Instruct)


config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen2.5-7B:   0%|          | 0/9 [00:00<?, ?it/s]


--- citation_single_source [PASS] ---
Q: Who created Python?
A: Python was created by Guido van Rossum [1].
[13 tokens, 1.6s]

--- citation_pick_right_source [PASS] ---
Q: What year was Python first released?
A: Python was first released to the public in 1991 [2].
[17 tokens, 1.7s]

--- conflicting_sources [PASS] ---
Q: Did the company's quarterly revenue increase?
A: The sources conflict. The Q2 Earnings Release indicates a 12% increase in quarterly revenue, while the Internal Filing Summary suggests a 5% decline. I don't have enough current information to answer this confidently based on the provided sources.
[52 tokens, 4.6s]

--- insufficient_information [PASS] ---
Q: What is the population of the newly founded city of Arcadia Falls?
A: I don't have enough current information to answer this confidently.
[13 tokens, 1.4s]

--- hallucination_resistance [PASS] ---
Q: Who is the CEO of Zylora Robotics?
A: I don't have enough current information to answer this confidently.
[13 tokens, 

,model,score_pct,avg_latency_sec,avg_tokens
1,Llama-3B,100.0,1.97,22.4
2,Llama-8B,100.0,2.26,20.0
4,Qwen2.5-3B,88.9,2.25,19.7
3,Qwen2.5-1.5B,77.8,2.61,30.1
5,Qwen2.5-7B,77.8,2.11,20.8
0,Llama-1B,44.4,1.99,24.1
